In [1]:
import numpy as np 
import pandas as pd
import tensorflow as tf 
import pickle
from tensorflow.keras.models import load_model

In [2]:
model=load_model('model.h5')

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('one_hot_encoder_geo.pkl','rb') as file:
    one_hot_encoder_geo = pickle.load(file)

2026-03-23 14:29:38.220447: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-03-23 14:29:38.220504: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-23 14:29:38.220516: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-23 14:29:38.220547: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-23 14:29:38.220569: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
input_data = {
    'CreditScore': 720,
    'Geography': 'Germany',
    'Gender': 'Female',
    'Age': 45,
    'Tenure': 6,
    'Balance': 120000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 0,
    'EstimatedSalary': 85000
}

In [5]:
# Convert input_data dict → DataFrame
input_df = pd.DataFrame([input_data])

# One-hot encode Geography
geo_encoded = one_hot_encoder_geo.transform(input_df[['Geography']]).toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=one_hot_encoder_geo.get_feature_names_out(['Geography'])
)

# Show encoded columns (optional)
geo_encoded_df

,Geography_France,Geography_Germany,Geography_India,Geography_Spain
0,0.0,1.0,0.0,0.0


In [6]:
# Drop original Geography column
input_df = input_df.drop('Geography', axis=1)

# Combine
input_df = pd.concat(
    [input_df.reset_index(drop=True), geo_encoded_df],
    axis=1
)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_India,Geography_Spain
0,720,Female,45,6,120000,2,1,0,85000,0.0,1.0,0.0,0.0


In [7]:
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_India,Geography_Spain
0,720,0,45,6,120000,2,1,0,85000,0.0,1.0,0.0,0.0


In [9]:
input_scaled=scaler.transform(input_df)
input_scaled

array([[ 0.8199145 , -1.        ,  0.09727322,  0.29053855,  0.36096395,
        -0.5653634 ,  1.05483461, -0.99335541,  0.04749098, -0.58761508,
         1.59041245, -0.531085  , -0.56195149]])

In [10]:
prediction=model.predict(input_scaled)
prediction

2026-03-23 14:31:47.086361: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 801ms/step


array([[0.5747276]], dtype=float32)

In [11]:
# Get probability
prediction_proba = prediction[0][0]

# Print probability
print(prediction_proba)

# Condition check
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

0.5747276
The customer is likely to churn.
